# CHI@Edge Coding Benchmark v4 — gold verification notebook

Runs the SWE-bench-style admission gate (every gold must pass its own checkers),
the computed tier assignment, and shows how to score a pasted model answer.

Offline by design: V1 availability items execute against the snapshot in `snapshots/`
via the stub `chi` package. **Caution:** scoring a model answer executes its code in a
subprocess for V1 items — run in a disposable environment.

In [ ]:
!python harness/validate_golds.py

In [ ]:
!python harness/tier_assign.py

## Score a model answer
Paste a model's raw response below (markdown with code fences is fine) and pick the item.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, 'harness')
from runner import evaluate, load_item

ITEM = 'P16'
ANSWER = '''
(paste the model answer here)
'''

rep = evaluate(load_item(Path(f'items/{ITEM}.yaml')), ANSWER,
               Path('snapshots/snapshot_synthetic_2026-07-04.json'))
print('ALL PASSED:', rep['all_passed'])
print('by group :', rep['groups'])
for r in rep['results']:
    print(('PASS' if r['passed'] else 'FAIL'), f"{r['check']}[{r['group']}]", '-', r['detail'])

## Batch-score a results directory
Layout: `runs/<condition>/<system>/<ITEM>.md` — one raw answer per file.

In [ ]:
from collections import defaultdict
import glob, os
rows = []
for path in sorted(glob.glob('runs/*/*/*.md')):
    _, cond, system, fname = path.split(os.sep)
    item_id = fname[:-3]
    rep = evaluate(load_item(Path(f'items/{item_id}.yaml')),
                   Path(path).read_text(),
                   Path('snapshots/snapshot_synthetic_2026-07-04.json'))
    rows.append(dict(item=item_id, condition=cond, system=system,
                     verdict='PASS' if rep['all_passed'] else 'FAIL',
                     groups=rep['groups']))
agg = defaultdict(lambda: [0, 0])
for r in rows:
    for g, v in r['groups'].items():
        agg[(r['condition'], r['system'], g)][0] += v['passed']
        agg[(r['condition'], r['system'], g)][1] += v['total']
print(json.dumps(rows, indent=1))
print('\nPer-group pass rates (the H0/H1 instrument):')
for k in sorted(agg):
    p, t = agg[k]
    print(k, f'{p}/{t}', f'{p/t:.0%}' if t else '-')